# 05 · Clasificar el tema de las noticias

Paso secundario y sin fine-tuning. Asigna a cada noticia una de las siete categorías de `tema` y compara seis combinaciones: TF-IDF y dos modelos de embeddings, cada uno con regresión logística y SVM lineal, ambos con pesos por clase.

La combinación se elige por F1 macro en `validation.csv`. `test.csv` se usa una sola vez, con la combinación ya elegida. El resultado no evalúa la veracidad de la noticia ni participa en la extracción de claims.


## Requisitos

Usa los mismos CSV cerrados del notebook 02. Los modelos de embeddings se descargan a la caché local de Hugging Face la primera vez.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

DATASET_NAME = 'claims_1500'
FINAL_DIR = ROOT / 'artifacts' / '02_conjuntos' / DATASET_NAME
OUTPUT_DIR = ROOT / 'artifacts' / '04_temas' / DATASET_NAME
SEED = 20261103
CACHE_DIR = None  # por ejemplo: '/ruta/a/cache_huggingface'
OVERWRITE_EXISTING = False  # test se evalúa una sola vez; no repetir tras mirar resultados

print({'dataset': DATASET_NAME, 'output_dir': str(OUTPUT_DIR)})


## Cargar los splits

Se reutiliza la validación de los notebooks de entrenamiento: tamaños exactos y ninguna noticia compartida entre splits.


In [ ]:
from claim_extraction.workflow import DATASET_SPLIT_SIZES, validate_sem1_splits

SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
splits = validate_sem1_splits(SPLIT_PATHS, expected_sizes=DATASET_SPLIT_SIZES[DATASET_NAME])
print({name: frame['tema'].value_counts().to_dict() for name, frame in splits.items()})


## Elegir la combinación y evaluar

La entrada es el título y el comienzo del cuerpo. TF-IDF se ajusta solo con train. En un empate gana la primera combinación en el orden declarado.


In [ ]:
import json

from claim_extraction.topics import select_and_evaluate_topic_classifier, sentence_transformer_encoder

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not OVERWRITE_EXISTING:
    raise FileExistsError(f'Ya existe una evaluación en {OUTPUT_DIR}. No se sobrescribe.')

results = select_and_evaluate_topic_classifier(
    splits['train'],
    splits['validation'],
    splits['test'],
    encoder=sentence_transformer_encoder(cache_dir=CACHE_DIR),
    seed=SEED,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results['validation_table'].to_csv(OUTPUT_DIR / 'comparacion_validacion.csv', index=False, encoding='utf-8')
results['per_class'].to_csv(OUTPUT_DIR / 'metricas_por_tema_test.csv', index=False, encoding='utf-8')
results['confusion'].to_csv(OUTPUT_DIR / 'matriz_confusion_test.csv', encoding='utf-8')
results['test_predictions'].to_csv(OUTPUT_DIR / 'predicciones_test.csv', index=False, encoding='utf-8')
(OUTPUT_DIR / 'resumen.json').write_text(
    json.dumps({'dataset': DATASET_NAME, 'seed': SEED, 'seleccionada': results['selected'], 'test': results['test_metrics']}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Seleccionada:', results['selected'])
print('Test:', results['test_metrics'])
results['validation_table'].sort_values('f1_macro', ascending=False)


In [ ]:
results['per_class']


In [ ]:
results['confusion']
